# Setup Agentic Information Retrieval (IR) Multi-Agent System
Questo notebook è configurato per l'esecuzione su **Google Colab** (consigliato: acceleratore **T4 GPU** in Runtime -> Cambia tipo di runtime).

Il sistema implementa un'architettura multi-agente ciclica basata su:
- **Adaptive-RAG** (Jeong et al., 2024): classificazione della complessità e routing dinamico tra Knowledge Base locale di Wikipedia e Web Search.
- **Self-RAG** (Asai et al., 2023): critique tokens (`is_context_relevant`, `is_query_answered`) per garantire il grounding ed evitare allucinazioni.
- **Reflexion** (Shinn et al., 2023): memoria episodica verbale per l'auto-correzione tra iterazioni successive.
- **Least-to-Most & Query2doc** (Zhou et al., 2022; Wang et al., 2023): scomposizione ed espansione semantica delle query nel Refiner.
- **ReAct & IRCoT** (Yao et al., 2022; Trivedi et al., 2022): recupero incrementale e ragionamento interleaved nel Planner.

In [ ]:
# 1. Verifica disponibilità GPU NVIDIA
!nvidia-smi
import os
if os.path.exists('/usr/lib64-nvidia'):
    os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')

# 2. Installazione runtime CUDA 12 per linking dinamico (risolve libcudart.so.12)
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12

# 3. Installazione librerie core di orchestrazione e ricerca
!pip install -q langchain langgraph langchain-community huggingface-hub pydantic datasets rank_bm25
!pip install -q -U ddgs duckduckgo-search beautifulsoup4 requests pandas

# 4. Installazione di llama-cpp-python precompilato con supporto CUDA 12 per GPU Colab
wheel_url = "https://abetlen.github.io/llama-cpp-python/whl/cu124"
!pip install llama-cpp-python --extra-index-url {wheel_url} -q


## Clonazione e Sincronizzazione della Repository
Scarichiamo o sincronizziamo il codice sorgente aggiornato della repository GitHub.

In [ ]:
import os, sys

REPO_URL = "https://github.com/GabCovone/AgenticAskMe"
repo_name = REPO_URL.split("/")[-1].replace(".git", "")

# Assicura che la directory di base sia /content su Google Colab
if os.path.exists("/content"):
    os.chdir("/content")

if not os.path.exists(repo_name):
    !git clone {REPO_URL}
    os.chdir(repo_name)
else:
    os.chdir(repo_name)
    !git fetch origin main
    !git reset --hard origin/main

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Directory di lavoro attiva: {os.getcwd()}")


## Inizializzazione della Knowledge Base Locale e del Grafo Multi-Agente
1. Inizializziamo il database SQLite locale (`wikipedia_local.db`) con indice full-text (FTS5) e BM25Okapi in cache.
2. Inizializziamo il modello LLM (`Mistral-Nemo-Instruct` o `Gemma-2-9B-it`) con **temperatura a 0.3** per garantire flessibilità generativa e precisione nello schema JSON.

In [ ]:
import warnings, logging, sys, importlib
warnings.filterwarnings('ignore')
logging.captureWarnings(True)
logging.getLogger('py.warnings').setLevel(logging.ERROR)

import state, local_kb, Planner, Refiner, Validator, graph
importlib.reload(state)
importlib.reload(local_kb)
importlib.reload(Planner)
importlib.reload(Refiner)
importlib.reload(Validator)
importlib.reload(graph)
from graph import build_graph

# 1. Inizializzazione immediata della Knowledge Base locale (SQLite FTS5 + BM25Okapi)
local_kb.init_db(force_reseed=False)
kb_info = local_kb.get_stats()
print(f"Knowledge Base locale attiva: {kb_info['total_articles']} articoli indicizzati in {kb_info['db_path']}.")

# 2. Compilazione del Grafo LangGraph
app = build_graph()
print("Grafo multi-agente compilato e pronto all'uso!")


## Test Interattivo su Query Singola
Testiamo una domanda per osservare l'intera traiettoria agentica:
- **Validator**: valutazione della complessità e routing (`route_to_refinement` o `route_to_planning`)
- **Refiner**: scomposizione Least-to-Most o espansione Query2doc
- **Planner**: Adaptive-RAG (scelta tra Local Wikipedia KB e Web Search) + ReAct multi-step
- **Validator**: controllo di aderenza Self-RAG (IsRel, IsSup) e verdetto finale

In [ ]:
import warnings, logging, sys, importlib
warnings.filterwarnings('ignore')
logging.captureWarnings(True)
logging.getLogger('py.warnings').setLevel(logging.ERROR)

import state, local_kb, Planner, Refiner, Validator, graph
importlib.reload(state)
importlib.reload(local_kb)
importlib.reload(Planner)
importlib.reload(Refiner)
importlib.reload(Validator)
importlib.reload(graph)
from graph import build_graph
app = build_graph()

# Inserisci qui una domanda di test:
test_query = "Were Scott Derrickson and Ed Wood of the same nationality?"

initial_state = {
    "original_query": test_query,
    "current_query": test_query,
    "retrieved_context": "",
    "num_refinement": 0,
    "num_planning": 0,
    "next_node": "",
    "feedback_history": [],
    "final_answer": ""
}

print(f"=== AVVIO TEST INTERATTIVO PER: '{test_query}' ===\n")
final_output = ""

for output in app.stream(initial_state, {"recursion_limit": 30}):
    for node_name, state_update in output.items():
        print(f"\n>>> Passaggio per il nodo: [{node_name.upper()}]")
        if "next_node" in state_update:
            print(f"    Routing verso: {state_update['next_node'].upper()}")
        if "current_query" in state_update and state_update["current_query"] != test_query:
            print(f"    Query raffinata: {state_update['current_query']}")
        if "final_answer" in state_update and state_update["final_answer"]:
            final_output = state_update["final_answer"]
            print(f"    Proposta risposta: {final_output}")

print("\n" + "="*60)
print(f"RISPOSTA FINALE SINTETIZZATA:\n{final_output}")
print("="*60)


## Benchmark Quantitativo su HotpotQA (Multi-Hop Evaluation)
Eseguiamo la valutazione automatica su un campione di domande multi-hop dal dataset gold-standard **HotpotQA** (`fullwiki`).

In [ ]:
import warnings, logging, sys, importlib
warnings.filterwarnings('ignore')
logging.captureWarnings(True)
logging.getLogger('py.warnings').setLevel(logging.ERROR)

import state, local_kb, Planner, Refiner, Validator, graph
importlib.reload(state)
importlib.reload(local_kb)
importlib.reload(Planner)
importlib.reload(Refiner)
importlib.reload(Validator)
importlib.reload(graph)
from graph import build_graph
app = build_graph()

from datasets import load_dataset
import time

print("Scaricamento del subset di validazione di HotpotQA...")
dataset = load_dataset("hotpotqa/hotpot_qa", "fullwiki", split="validation")

NUM_TESTS = 10  # Numero di campioni da testare nel benchmark
correct = 0
results_log = []

# CONFIGURAZIONE MODALITÀ RETRIEVAL (Zero modifiche ai prompt):
# - 'full': Sistema Integrale (KB Locale + Ricerca Web Live)
# - 'baseline': Sistema Baseline (Solo Knowledge Base Locale di Wikipedia)
RETRIEVAL_MODE = "full"  # Imposta 'baseline' per vincolare la ricerca alla sola KB locale
Planner.set_retrieval_mode(RETRIEVAL_MODE)
print(f"Modalità di Retrieval Configurata: {Planner.get_retrieval_mode().upper()}")

print(f"\n--- AVVIO BENCHMARK SU {NUM_TESTS} DOMANDE HOTPOTQA --- ")
for i in range(NUM_TESTS):
    sample = dataset[i]
    question = sample["question"]
    gold_answer = sample["answer"]
    
    print(f"\n[Test {i+1}/{NUM_TESTS}]")
    print(f"Domanda: {question}")
    print(f"Gold Answer: {gold_answer}")
    
    state = {
        "original_query": question,
        "current_query": question,
        "retrieved_context": "",
        "num_refinement": 0,
        "num_planning": 0,
        "next_node": "",
        "feedback_history": [],
        "final_answer": "",
        "retrieval_mode": RETRIEVAL_MODE
    }
    
    final_answer = ""
    last_state = {}
    try:
        for output in app.stream(state, {"recursion_limit": 30}):
            for key, value in output.items():
                if isinstance(value, dict):
                    last_state.update(value)
                if "final_answer" in value and value["final_answer"]:
                    final_answer = value["final_answer"]
    except Exception as e:
        print(f"Errore durante l'esecuzione: {e}")
        final_answer = "Error"
        
    print(f"Risposta Agente: {final_answer}")
    
    # Calcolo metriche standard di Information Retrieval & Question Answering (HotpotQA / SQuAD)
    import re, string, collections
    def normalize_qa_text(s):
        def remove_articles(text):
            return re.sub(r'\b(a|an|the)\b', ' ', text)
        def white_space_fix(text):
            return ' '.join(text.split())
        def remove_punc(text):
            exclude = set(string.punctuation)
            return ''.join(ch for ch in text if ch not in exclude)
        return white_space_fix(remove_articles(remove_punc(s.lower())))

    norm_gold = normalize_qa_text(gold_answer)
    norm_pred = normalize_qa_text(final_answer)

    if norm_gold in ['yes', 'no']:
        opp = 'no' if norm_gold == 'yes' else 'yes'
        pred_words = norm_pred.split()
        is_polar_match = (norm_gold in pred_words) and (opp not in pred_words)
        q_f1 = 1.0 if is_polar_match else 0.0
        q_prec = 1.0 if is_polar_match else 0.0
        q_rec = 1.0 if is_polar_match else 0.0
        q_em = 1.0 if is_polar_match else 0.0
        is_correct = is_polar_match
    else:
        g_tokens = norm_gold.split()
        p_tokens = norm_pred.split()
        common_tokens = collections.Counter(g_tokens) & collections.Counter(p_tokens)
        num_same = sum(common_tokens.values())
        if num_same == 0:
            q_f1, q_prec, q_rec = 0.0, 0.0, 0.0
        else:
            q_prec = 1.0 * num_same / len(p_tokens)
            q_rec = 1.0 * num_same / len(g_tokens)
            q_f1 = (2 * q_prec * q_rec) / (q_prec + q_rec)
        before_than = final_answer.split(' than ')[0] if (' or ' in question.lower() and ' than ' in final_answer.lower()) else final_answer
        exact_sub = bool(re.search(rf'(?<!\w){re.escape(gold_answer.strip().lower())}(?!\w)', before_than.lower()))
        q_em = 1.0 if (norm_gold == norm_pred or norm_gold in norm_pred) else 0.0
        is_correct = exact_sub or (q_f1 >= 0.5)

    if is_correct:
        print(f'Valutazione: CORRETTA [OK] (F1={q_f1:.2f}, EM={q_em:.0f})')
        correct += 1
    else:
        print(f'Valutazione: ERRATA [MISMATCH] (F1={q_f1:.2f}, EM={q_em:.0f})')

    results_log.append({
        'question': question,
        'gold': gold_answer,
        'predicted': final_answer,
        'correct': is_correct,
        'f1': q_f1,
        'precision': q_prec,
        'recall': q_rec,
        'em': q_em
    })

    # Persistenza opzionale nel database storico locale SQLite
    try:
        import history_db
        history_db.log_query_run(
            query=question,
            final_answer=final_answer,
            gold_answer=gold_answer,
            is_correct=is_correct,
            f1=q_f1,
            precision=q_prec,
            recall=q_rec,
            em=q_em,
            accuracy=1.0 if is_correct else 0.0,
            num_refinement=last_state.get("refinement_count", last_state.get("num_refinement", 0)),
            num_planning=last_state.get("planning_hops", last_state.get("num_planning", 0)),
            steps_trace=last_state.get("visited_nodes", []),
            reasoning=last_state.get("reasoning", ""),
            retrieved_context=last_state.get("retrieved_context", ""),
            system_mode=RETRIEVAL_MODE
        )
    except Exception:
        pass

    time.sleep(1) # Rispetto dei limiti di frequenza per la ricerca web

mean_prec = (sum(r['precision'] for r in results_log) / NUM_TESTS) * 100
mean_rec = (sum(r['recall'] for r in results_log) / NUM_TESTS) * 100
macro_f1 = (sum(r['f1'] for r in results_log) / NUM_TESTS) * 100
mean_em = (sum(r['em'] for r in results_log) / NUM_TESTS) * 100
accuracy = (correct / NUM_TESTS) * 100

print('\n' + '='*65)
print(f'RISULTATI FINALI BENCHMARK HOTPOTQA ({NUM_TESTS} DOMANDE)')
print('='*65)
print(f'  * Grounded Accuracy (Match): {accuracy:.1f}% ({correct}/{NUM_TESTS})')
print(f'  * Macro F1-Score:             {macro_f1:.2f}%')
print(f'  * Exact Match (EM):           {mean_em:.1f}%')
print(f'  * Mean Precision:             {mean_prec:.2f}%')
print(f'  * Mean Recall:                {mean_rec:.2f}%')
print('='*65 + '\n')


In [ ]:
# =====================================================================
# ESPORTAZIONE E DOWNLOAD DEL DATABASE E DEL FILE JSON
# =====================================================================
# Esporta tutti i test eseguiti (inclusa la modalità 'system_mode') e scarica lo zip

import sqlite3, json, zipfile
from google.colab import files

# 1. Estrai tutti i record da agentic_history.db
conn = sqlite3.connect("agentic_history.db")
conn.row_factory = sqlite3.Row
rows = [dict(r) for r in conn.cursor().execute("SELECT * FROM query_runs ORDER BY id ASC").fetchall()]
conn.close()

# 2. Salva in formato JSON con system_mode
with open("benchmark_history.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

# 3. Crea l'archivio ZIP con DB e JSON per evitare blocchi del browser
with zipfile.ZipFile("benchmark_results.zip", "w") as zipf:
    zipf.write("agentic_history.db")
    zipf.write("benchmark_history.json")

print(f"Archiviati con successo {len(rows)} record (modalità: full/baseline).")
print("Avvio download automatico di 'benchmark_results.zip'...")
files.download("benchmark_results.zip")


In [ ]:
# =====================================================================
# AVVIO DELLA DASHBOARD STREAMLIT IN GOOGLE COLAB (Tunnel Diretto)
# =====================================================================
# 1. Installa Streamlit e LocalTunnel per tunneling sicuro con WebSockets
!pip install -q streamlit
!npm install -g localtunnel > /dev/null

import subprocess, sys, time, urllib.request

# 2. Recupera l'IP pubblico di Colab (necessario come password su LocalTunnel)
try:
    colab_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
except Exception:
    colab_ip = "N/A"

# 3. Avvia Streamlit disabilitando CORS e XSRF (indispensabili su Colab)
process = subprocess.Popen([
    sys.executable, "-m", "streamlit", "run", "Dashboard.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])
time.sleep(4)

print("\n" + "="*65)
print(f"🔑 PASSWORD LOCALTUNNEL (Inserisci questo IP): {colab_ip}")
print("="*65)
print("Clicca sul link sottostante ('your url is: ...') per aprire la Dashboard:\n")

# 4. Espone la porta su URL pubblico
!npx localtunnel --port 8501
